In [34]:
import os
import json
import torch
from PIL import Image
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor
DATA_DIR = "../data"
FEATURES_DIR = "../features"
os.makedirs(FEATURES_DIR, exist_ok=True)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [35]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [36]:
# Load all splits
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]
train_data = load_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
dev_data = load_jsonl(os.path.join(DATA_DIR, "dev.jsonl"))
test_data = load_jsonl(os.path.join(DATA_DIR, "test.jsonl"))

In [37]:
print(f"Train: {len(train_data)}, Dev: {len(dev_data)}, Test: {len(test_data)}")

Train: 8500, Dev: 500, Test: 1000


In [38]:
from PIL import Image as PILImage

s = train_data[0]
print(s)

img_path = os.path.join(DATA_DIR, s['img'])
img = PILImage.open(img_path)
text = s['text']



input = clip_processor(img, text, return_tensors="pt", padding=True, truncation=True)
img_features = clip_model.get_image_features(input.pixel_values.to(device))
print(img_features)




{'id': 42953, 'img': 'img/42953.png', 'label': 0, 'text': 'its their character not their color that matters'}
BaseModelOutputWithPooling(last_hidden_state=tensor([[[ 0.4367, -0.0517,  0.0630,  ...,  0.3479, -0.0958, -0.0994],
         [-0.0866, -0.0964, -0.2573,  ...,  0.1289,  0.1387,  0.0418],
         [-0.2712,  0.2414, -0.0299,  ...,  0.3818,  0.0980,  0.0745],
         ...,
         [-0.0891,  0.3077,  0.2498,  ...,  0.2800,  0.0187,  0.0247],
         [-0.2232, -0.0253,  1.1790,  ...,  0.3135,  0.4741, -0.2988],
         [ 0.1438,  0.2122,  1.0259,  ...,  0.0626,  0.2813,  0.1173]]],
       device='mps:0', grad_fn=<AddBackward0>), pooler_output=tensor([[ 2.0777e-01, -2.2538e-01, -5.1436e-02,  2.7912e-01, -1.4497e-01,
         -4.1064e-01,  1.4503e-01,  5.8678e-01,  1.0786e+00,  2.5907e-01,
          6.7167e-02,  1.2589e-01, -5.1857e-01,  7.6112e-02,  3.5065e-01,
          3.0589e-01,  1.4206e+00,  1.8601e-02, -2.9354e-01,  2.2423e-01,
          3.7405e-01,  1.5356e-01,  1.1642e-0

In [39]:
def extract_features(data, batch_size=32):
    """Extract CLIP image and text features for a list of samples."""
    all_img_features = []
    all_txt_features = []
    all_labels = []
    all_ids = []
    for i in tqdm(range(0, len(data), batch_size)):
        batch = data[i:i+batch_size]
        # Load images
        images = []
        texts = []
        for sample in batch:
            img_path = os.path.join(DATA_DIR, sample['img'])
            img = Image.open(img_path).convert('RGB')
            images.append(img)
            texts.append(sample['text'])
        # Process through CLIP
        inputs = clip_processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        # Move to device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        # Extract features (no gradients needed — we're not training CLIP here)
        with torch.no_grad():
            img_features = clip_model.get_image_features(pixel_values=inputs['pixel_values'])                                
            txt_features = clip_model.get_text_features(input_ids=inputs['input_ids'],                                       
                                                          =inputs['attention_mask'])
                                                                                                                             
            # If output is not a tensor, extract the pooler_output                                                           
            if not isinstance(img_features, torch.Tensor):
                img_features = img_features.pooler_output                                                                    
            if not isinstance(txt_features, torch.Tensor):
                txt_features = txt_features.pooler_output
        # Normalize to unit length
        img_features = img_features / img_features.norm(dim=-1, keepdim=True)
        txt_features = txt_features / txt_features.norm(dim=-1, keepdim=True)
        # Move back to CPU for storage
        all_img_features.append(img_features.cpu())
        all_txt_features.append(txt_features.cpu())
        all_labels.extend([s.get('label', -1) for s in batch])

        all_ids.extend([s['id'] for s in batch])
    return {
        'img_features': torch.cat(all_img_features, dim=0),
        'txt_features': torch.cat(all_txt_features, dim=0),
        'labels': torch.tensor(all_labels, dtype=torch.float32),
        'ids': all_ids
    }

SyntaxError: invalid syntax (1638514078.py, line 31)

In [ ]:
print("Extracting train features...")
train_features = extract_features(train_data)
print(f"Train — img: {train_features['img_features'].shape}, txt: {train_features['txt_features'].shape}")
                                                                                                          
print("\nExtracting dev features...")                                                                                        
dev_features = extract_features(dev_data)                                                                                    
print(f"Dev — img: {dev_features['img_features'].shape}, txt: {dev_features['txt_features'].shape}")                         
                                                                                                                             
print("\nExtracting test features...")                                                                                     
test_features = extract_features(test_data)
print(f"Test — img: {test_features['img_features'].shape}, txt: {test_features['txt_features'].shape}")


Extracting train features...


100%|██████████| 266/266 [02:04<00:00,  2.14it/s]


Train — img: torch.Size([8500, 512]), txt: torch.Size([8500, 512])

Extracting dev features...


100%|██████████| 16/16 [00:07<00:00,  2.21it/s]


Dev — img: torch.Size([500, 512]), txt: torch.Size([500, 512])

Extracting test features...


100%|██████████| 32/32 [00:14<00:00,  2.14it/s]

Test — img: torch.Size([1000, 512]), txt: torch.Size([1000, 512])


In [ ]:
torch.save({
    'train': train_features,
    'dev': dev_features,
    'test': test_features
  }, os.path.join(FEATURES_DIR, "clip_features.pt"))

In [ ]:
size_mb = os.path.getsize(os.path.join(FEATURES_DIR, "clip_features.pt")) / (1024*1024)
print(f"Saved to features/clip_features.pt ({size_mb:.1f} MB)")

Saved to features/clip_features.pt (39.1 MB)


In [ ]:
loaded = torch.load(os.path.join(FEATURES_DIR, "clip_features.pt"))
print(f"Train features shape: {loaded['train']['img_features'].shape}")
print(f"First image feature (first 5 dims): {loaded['train']['img_features'][0][:5]}")
print(f"Feature norm: {loaded['train']['img_features'][0].norm().item():.4f}")  # should be ~1.0

Train features shape: torch.Size([8500, 512])
First image feature (first 5 dims): tensor([ 0.0205, -0.0222, -0.0051,  0.0275, -0.0143])
Feature norm: 1.0000
